In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)

try:
    ROOT = Path(__file__).resolve().parent.parent
except NameError:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PATH = ROOT / "dataset" / "kosovo_overall_biodiversity.csv"
PROCESSED_DIR = ROOT / "processed"
REPORTS_DIR = ROOT / "reports"
CLEANING_LOG_PATH = PROCESSED_DIR / "cleaning_log.csv"
MISSING_REPORT_PATH = PROCESSED_DIR / "missing_report.csv"
ISSUE_LIST_PATH = REPORTS_DIR / "02_issue_list.csv"

PROCESSED_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)


def section(title):
    print(f"\n{'=' * 70}\n{title}\n{'=' * 70}")

# 02 - Integrimi, pastrimi dhe vlerat e zbrazëta

## Shablloni i dokumentimit dhe cleaning log

In [2]:
LOG_COLUMNS = [
    "hapi",
    "input",
    "rreshta_para",
    "kolona_para",
    "rreshta_pas",
    "kolona_pas",
    "arsyeja",
    "output",
    "commit",
]


def init_cleaning_log(path=CLEANING_LOG_PATH):
    """Krijon cleaning_log.csv bosh me kolonat standarde, nëse nuk ekziston."""
    if not path.exists():
        pd.DataFrame(columns=LOG_COLUMNS).to_csv(path, index=False)
    return path


def log_step(
    hapi,
    inputi,
    shape_para,
    shape_pas,
    arsyeja,
    outputi="",
    commit="pending",
    path=CLEANING_LOG_PATH,
):
    """Shton një rresht në cleaning_log.csv për një hap pastrimi."""
    init_cleaning_log(path)
    row = {
        "hapi": hapi,
        "input": inputi,
        "rreshta_para": shape_para[0],
        "kolona_para": shape_para[1],
        "rreshta_pas": shape_pas[0],
        "kolona_pas": shape_pas[1],
        "arsyeja": arsyeja,
        "output": outputi,
        "commit": commit,
    }
    log = pd.read_csv(path)
    log = pd.concat([log, pd.DataFrame([row])], ignore_index=True)
    log.to_csv(path, index=False)
    return log.tail(1)

In [3]:
section("Shablloni i dokumentimit dhe cleaning log")
init_cleaning_log()
print(pd.read_csv(CLEANING_LOG_PATH))


Shablloni i dokumentimit dhe cleaning log
Empty DataFrame
Columns: [hapi, input, rreshta_para, kolona_para, rreshta_pas, kolona_pas, arsyeja, output, commit]
Index: []


### Dokumentimi i hapit: Shablloni i dokumentimit dhe cleaning log

1. Input: Struktura e repo-s; folderët notebooks/, processed/, reports/;
   skedari i planifikuar processed/cleaning_log.csv.
2. Çka u bë: U krijua shablloni i dokumentimit dhe funksionet
   init_cleaning_log() / log_step() për regjistrimin e hapave të pastrimit.
3. Rreshta para / rreshta pas: Dataset-i nuk u ndryshua.
   cleaning_log.csv: 0 rreshta, 9 kolona pas inicializimit.
4. Pse u bë: Për të pasur gjurmë të qartë të çdo vendimi dhe për ta bërë
   punën të kontrollueshme nga profesori/grupi.
5. Output: processed/cleaning_log.csv bosh me kolonat standarde.
6. Commit: Add cleaning documentation template and log helper.

## Eksplorimi fillestar i strukturës

In [4]:
section("Eksplorimi fillestar i strukturës")
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
rows_before, cols_before = df_raw.shape
print(f"Rreshta: {rows_before:,}")
print(f"Kolona: {cols_before:,}")
print(df_raw.head())

df_raw.info(memory_usage="deep")

dtypes_report = (
    df_raw.dtypes.astype(str)
    .rename("tipi_i_lexuar")
    .reset_index()
    .rename(columns={"index": "kolona"})
)
print("\nTipet e lexuara:")
print(dtypes_report.to_string(index=False))

five_values_per_column = pd.DataFrame({
    "kolona": df_raw.columns,
    "5_vlera_shembull": [
        df_raw[c].dropna().astype(str).drop_duplicates().head(5).tolist()
        for c in df_raw.columns
    ],
    "vlera_unike": [df_raw[c].nunique(dropna=True) for c in df_raw.columns],
})
print("\n5 vlera shembull nga çdo kolonë:")
print(five_values_per_column.to_string(index=False))

issue_rows = []

date_patterns = (
    df_raw["eventDate"]
    .dropna()
    .astype(str)
    .str.replace(r"\d", "9", regex=True)
    .value_counts()
)
if len(date_patterns) > 1:
    issue_rows.append({
        "kategoria": "Format datash",
        "kolona": "eventDate",
        "problemi": f"eventDate paraqitet në {len(date_patterns)} formate/pattern-e të ndryshme.",
        "shembuj": "; ".join(date_patterns.head(5).index.tolist()),
        "veprim_i_sugjeruar": (
            "Parsimi me pd.to_datetime pas ndarjes së intervaleve dhe ruajtja e "
            "precision/date-range flag."
        ),
    })

for c in df_raw.columns:
    values = set(df_raw[c].dropna().astype(str).str.upper().unique())
    if values and values.issubset({"TRUE", "FALSE"}):
        issue_rows.append({
            "kategoria": "Boolean TRUE/FALSE",
            "kolona": c,
            "problemi": "Kolona përmban vlera TRUE/FALSE që duhen standardizuar.",
            "shembuj": ", ".join(sorted(values)),
            "veprim_i_sugjeruar": "Konvertim në boolean ose 0/1 në fazën e pastrimit/binarizimit.",
        })

for c in ["municipality", "district", "protectedArea"]:
    count = df_raw[c].fillna("").astype(str).str.contains("/", regex=False).sum()
    if count > 0:
        examples = (
            df_raw.loc[df_raw[c].fillna("").astype(str).str.contains("/", regex=False), c]
            .dropna()
            .drop_duplicates()
            .head(5)
            .tolist()
        )
        issue_rows.append({
            "kategoria": "Emra dygjuhësh",
            "kolona": c,
            "problemi": f"{count} rreshta përmbajnë emra me ndarës /.",
            "shembuj": "; ".join(examples),
            "veprim_i_sugjeruar": (
                "Ruaj vlerën origjinale dhe, nëse nevojitet, krijo kolonë të "
                "standardizuar për analizë."
            ),
        })

for c in df_raw.columns:
    count = df_raw[c].fillna("").astype(str).str.contains(";", regex=False).sum()
    if count > 0:
        examples = (
            df_raw.loc[df_raw[c].fillna("").astype(str).str.contains(";", regex=False), c]
            .dropna()
            .drop_duplicates()
            .head(3)
            .tolist()
        )
        issue_rows.append({
            "kategoria": "Lista në një kolonë",
            "kolona": c,
            "problemi": f"{count} rreshta kanë disa vlera të ndara me ;.",
            "shembuj": "; ".join(examples),
            "veprim_i_sugjeruar": "Split + explode për raportim ose krijim flamujsh binarë.",
        })

issue_list = pd.DataFrame(issue_rows)
issue_list.to_csv(ISSUE_LIST_PATH, index=False)
print("\nLista e problemeve të vërejtura:")
print(issue_list.to_string(index=False))


Eksplorimi fillestar i strukturës
Rreshta: 46,831
Kolona: 42
       gbifID                     scientificName           species             vernacularName   kingdom        phylum          class           order          family     genus taxonRank   taxonKey  speciesKey  \
0  6500474327     Mantis religiosa (Linne, 1758)  Mantis religiosa             Praying Mantis  Animalia    Arthropoda        Insecta        Mantodea        Mantidae    Mantis   SPECIES  6258028.0   6258028.0   
1  6500462703  Issoria lathonia (Linnaeus, 1758)  Issoria lathonia  Queen of Spain Fritillary  Animalia    Arthropoda        Insecta     Lepidoptera     Nymphalidae   Issoria   SPECIES  1905383.0   1905383.0   
2  6500450749  Podarcis muralis (Laurenti, 1768)  Podarcis muralis         Common Wall Lizard  Animalia      Chordata       Squamata             NaN      Lacertidae  Podarcis   SPECIES  2469188.0   2469188.0   
3  6500428815              Armeria alpina Willd.    Armeria alpina            Alpen-Grasnelke 

<class 'pandas.DataFrame'>
RangeIndex: 46831 entries, 0 to 46830
Data columns (total 42 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   gbifID                         46831 non-null  int64  
 1   scientificName                 46794 non-null  str    
 2   species                        45618 non-null  str    
 3   vernacularName                 42958 non-null  str    
 4   kingdom                        46794 non-null  str    
 5   phylum                         46762 non-null  str    
 6   class                          46443 non-null  str    
 7   order                          46497 non-null  str    
 8   family                         46673 non-null  str    
 9   genus                          46636 non-null  str    
 10  taxonRank                      46794 non-null  str    
 11  taxonKey                       46794 non-null  float64
 12  speciesKey                     45618 non-null  float64
 1


Lista e problemeve të vërejtura:
          kategoria            kolona                                                 problemi                                                                                                                                                                                                                                                                                                                                       shembuj                                                                            veprim_i_sugjeruar
      Format datash         eventDate eventDate paraqitet në 11 formate/pattern-e të ndryshme.                                                                                                                                                                                                                                                             9999-99-99; 9999-99-99T99:99:99; 9999-99-99T99:99; 9999-99-99T99:99:99.999Z; 9999 Parsimi me pd.

### Dokumentimi i hapit: Eksplorimi fillestar i strukturës

1. Input: dataset/kosovo_overall_biodiversity.csv; të gjitha kolonat për
   shape, info, dtypes dhe 5 vlera shembull.
2. Çka u bë: U kontrollua struktura fillestare e dataset-it dhe u krijua
   lista e problemeve të vërejtura: data në formate të ndryshme, boolean
   TRUE/FALSE, emra dygjuhësh dhe lista me ;.
3. Rreshta para / rreshta pas: Dataset-i nuk u ndryshua; df_raw.shape mbetet
   i njëjtë para/pas eksplorimit.
4. Pse u bë: Për të kuptuar strukturën reale para çdo pastrimi dhe për të
   shmangur vendime pa evidencë.
5. Output: reports/02_issue_list.csv, dtypes_report,
   five_values_per_column, issue_list.
6. Commit: Add initial structure exploration and issue list.

## Inventari i vlerave të zbrazëta

In [5]:
section("Inventari i vlerave të zbrazëta")
missing_report = pd.DataFrame({
    "kolona": df_raw.columns,
    "mungojne": df_raw.isna().sum().values,
    "perqindja": (df_raw.isna().mean().values * 100).round(2),
    "vlera_te_plotesuara": df_raw.notna().sum().values,
}).sort_values(["perqindja", "kolona"], ascending=[False, True])

missing_report.to_csv(MISSING_REPORT_PATH, index=False)
print(missing_report.to_string(index=False))

print("\nKolonat më problematike për planin e pastrimit:")
print(missing_report[missing_report["mungojne"] > 0].head(15).to_string(index=False))


Inventari i vlerave të zbrazëta
                       kolona  mungojne  perqindja  vlera_te_plotesuara
               iasAppliesFrom     46807      99.95                   24
           establishmentMeans     46591      99.49                  240
                 identifiedBy     39096      83.48                 7735
                    elevation     37011      79.03                 9820
coordinateUncertaintyInMeters     32004      68.34                14827
                protectedArea     25762      55.01                21069
     protectedAreaDesignation     25762      55.01                21069
              individualCount     17697      37.79                29134
                     locality     15251      32.57                31580
               collectionCode     13469      28.76                33362
                catalogNumber     12559      26.82                34272
               vernacularName      3873       8.27                42958
                   recordedBy  

### Dokumentimi i hapit: Inventari i vlerave të zbrazëta

1. Input: dataset/kosovo_overall_biodiversity.csv; të gjitha kolonat.
2. Çka u bë: U llogarit numri dhe përqindja e NaN për çdo kolonë dhe raporti
   u rendit nga kolona më e zbrazët.
3. Rreshta para / rreshta pas: Dataset-i nuk u ndryshua. Raporti ka një
   rresht për çdo kolonë të dataset-it; kolonat e raportit janë kolona,
   mungojne, perqindja, vlera_te_plotesuara.
4. Pse u bë: Për të vendosur strategjinë e trajtimit të vlerave të zbrazëta
   në hapat e pastrimit.
5. Output: processed/missing_report.csv dhe tabela missing_report.
6. Commit: Add missing values inventory.

## Mungesat e fshehura dhe elevation = 0

Leximi i dytë përdor `keep_default_na=False` dhe `na_filter=False`, që `NA`, `null` dhe fushat boshe të mos humben gjatë leximit. Kontrollohen të gjitha kolonat si tekst, me përputhje të plotë pas heqjes së hapësirave dhe normalizimit të shkronjave. Vlera si `N/A` brenda një përshkrimi nuk klasifikohen automatikisht si mungesë.

Raportohen veçmas fushat boshe, vetëm hapësirat, tokenët dhe mungesat që pandas i ka njohur tashmë. Maskat efektive përdorin bashkimin e tyre, pa numërim të dyfishtë. `NE`, `DD`, `False` dhe zero në kolona të tjera nuk janë tokenë mungese.

Burimi gjeografik: [Agjencia e Statistikave të Kosovës, Some Facts on the Environment 2021–2022, f. 10](https://askapi.rks-gov.net/Custom/d9b7ae43-c68e-49a4-b713-5ec33c3a9c25.pdf#page=10), minimumi 270 m. Ky është kontroll plausibiliteti, jo zëvendësim i lartësisë për secilën pikë.

In [6]:
section("Mungesat e fshehura")
from IPython.display import Markdown, display

literal = pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False, na_filter=False)
assert literal.shape == df_raw.shape
assert literal["gbifID"].tolist() == df_raw["gbifID"].astype(str).tolist()
token_rows = []
token_mask = pd.DataFrame(False, index=df_raw.index, columns=df_raw.columns)
for column in literal:
    values = literal[column]
    stripped = values.str.strip()
    masks = {"empty_string": values.eq(""), "whitespace_only": values.ne("") & stripped.eq("")}
    masks.update({token: stripped.str.casefold().eq(token) for token in ["unknown", "na", "n/a", "null", "-"]})
    for token, mask in masks.items():
        token_rows.append({"kolona": column, "token": token, "numri": int(mask.sum()),
                           "perqindja": round(mask.mean() * 100, 2),
                           "njohur_si_nan": int((mask & df_raw[column].isna()).sum()),
                           "mungesa_te_reja": int((mask & df_raw[column].notna()).sum())})
        token_mask[column] |= mask
hidden_report = pd.DataFrame(token_rows)
hidden_report.to_csv(REPORTS_DIR / "hidden_missing_values.csv", index=False)

zero_elevation = pd.to_numeric(literal["elevation"], errors="coerce").eq(0)
effective_missing = df_raw.isna() | token_mask
effective_missing["elevation"] |= zero_elevation
effective_report = pd.DataFrame({
    "kolona": df_raw.columns,
    "nan_fillestar": df_raw.isna().sum().values,
    "token_shtese": (token_mask & df_raw.notna()).sum().values,
    "zero_elevation": [int(zero_elevation.sum()) if c == "elevation" else 0 for c in df_raw],
    "mungesa_efektive": effective_missing.sum().values,
    "perqindja_efektive": (effective_missing.mean().values * 100).round(2),
})
effective_report.to_csv(REPORTS_DIR / "effective_missing_report.csv", index=False)
display(hidden_report[hidden_report["numri"] > 0])
display(effective_report.sort_values("perqindja_efektive", ascending=False))

# Çelësi i burimit mbetet datasetKey; cache-i lokal shërben vetëm për etiketat.
cache_path = PROCESSED_DIR / "gbif_datasets_cache.csv"
source_titles = {}
if cache_path.exists():
    source_cache = pd.read_csv(cache_path)
    assert source_cache["datasetKey"].is_unique
    source_titles = source_cache.set_index("datasetKey")["datasetTitle"].to_dict()

def group_labels(column):
    return literal[column].str.strip().mask(effective_missing[column], "[mungon]")

zero_tables = []
for grouping in ["datasetKey", "basisOfRecord", "district", "municipality", "locality"]:
    frame = pd.DataFrame({"grupi": group_labels(grouping), "zero": zero_elevation,
                          "elevation_present": df_raw["elevation"].notna()})
    table = frame.groupby("grupi", dropna=False).agg(
        n=("zero", "size"), zero_count=("zero", "sum"), elevation_present=("elevation_present", "sum")
    ).reset_index()
    table.insert(0, "grupimi", grouping)
    table["zero_pct_group"] = (table["zero_count"] / table["n"] * 100).round(2)
    table["zero_pct_present"] = (table["zero_count"] / table["elevation_present"].replace(0, np.nan) * 100).round(2)
    table["datasetTitle"] = table["grupi"].map(source_titles) if grouping == "datasetKey" else ""
    zero_tables.append(table)
zero_groups = pd.concat(zero_tables, ignore_index=True)
zero_groups.to_csv(REPORTS_DIR / "elevation_zero_by_group.csv", index=False)
zero_locations = df_raw.loc[zero_elevation, ["gbifID", "datasetKey", "basisOfRecord", "district", "municipality", "locality", "decimalLatitude", "decimalLongitude", "elevation"]]
zero_locations.to_csv(REPORTS_DIR / "elevation_zero_records.csv", index=False)
display(zero_groups[(zero_groups["zero_count"] > 0) & (zero_groups["grupimi"] != "locality")])

n_zero = int(zero_elevation.sum())
n_present = int(df_raw["elevation"].notna().sum())
top_source = zero_groups[zero_groups["grupimi"].eq("datasetKey")].sort_values("zero_count", ascending=False).iloc[0]
elevation_decision = f"""# Vendimi për elevation = 0

Ka {n_zero:,} zero ({n_zero / len(df_raw) * 100:.2f}% të rreshtave; {n_zero / n_present * 100:.2f}% të {n_present:,} lartësive të plotësuara).
Burimi kryesor është {top_source['datasetTitle']} (`{top_source['grupi']}`), me {int(top_source['zero_count']):,} zero.
Zerot shfaqen në {zero_locations['district'].nunique()} rajone dhe {zero_locations['municipality'].nunique()} komuna; nuk përfaqësojnë një vend të vetëm të ulët.
Tabelat `elevation_zero_by_group.csv` japin numrin dhe emëruesin sipas burimit, basisOfRecord, rajonit, komunës dhe locality; `elevation_zero_records.csv` ruan koordinatat dhe ID-të për auditim.

**Decision + Rationale:** 0 m nuk trajtohet si lartësi reale e sipërfaqes në Kosovë. ASK raporton minimum 270 m në luginën e Drinit të Bardhë ([Some Facts on the Environment 2021–2022, f. 10](https://askapi.rks-gov.net/Custom/d9b7ae43-c68e-49a4-b713-5ec33c3a9c25.pdf#page=10)). Përqendrimi te një burim sugjeron placeholder ose problem eksportimi, por pa metadata të botuesit nuk mund të provohet shkaku për çdo rresht.

Në analizën e mungesave zero përfshihet në maskën efektive; në pastrimin e ardhshëm rekomandohet NaN dhe flamur `elevation_was_zero`, duke ruajtur origjinalin. Nuk zëvendësohet me 270 m ose me mesatare. Mungesa efektive bëhet {effective_missing['elevation'].sum():,} ({effective_missing['elevation'].mean() * 100:.2f}%).

Kufizime: nuk është bërë kontroll GIS/DEM për çdo pikë; lartësitë e tjera të dyshimta (përfshirë maksimumin {df_raw['elevation'].max():g} m) kërkojnë kontroll të veçantë. Referenca kombëtare nuk është prag automatik për të fshirë të gjitha vlerat nën 270 m. Këtu krijohen vetëm raporte, jo dataset i pastruar.
"""
(REPORTS_DIR / "elevation_zero_decision.md").write_text(elevation_decision, encoding="utf-8")
display(Markdown(elevation_decision))


Mungesat e fshehura


,kolona,token,numri,perqindja,njohur_si_nan,mungesa_te_reja
7,scientificName,empty_string,37,0.08,37,0
14,species,empty_string,1213,2.59,1213,0
21,vernacularName,empty_string,3873,8.27,3873,0
28,kingdom,empty_string,37,0.08,37,0
35,phylum,empty_string,69,0.15,69,0
42,class,empty_string,388,0.83,388,0
49,order,empty_string,334,0.71,334,0
56,family,empty_string,158,0.34,158,0
63,genus,empty_string,195,0.42,195,0
70,taxonRank,empty_string,37,0.08,37,0


,kolona,nan_fillestar,token_shtese,zero_elevation,mungesa_efektive,perqindja_efektive
41,iasAppliesFrom,46807,0,0,46807,99.95
31,establishmentMeans,46591,0,0,46591,99.49
22,elevation,37011,0,3634,40645,86.79
33,identifiedBy,39096,1,0,39097,83.49
21,coordinateUncertaintyInMeters,32004,0,0,32004,68.34
26,protectedArea,25762,0,0,25762,55.01
27,protectedAreaDesignation,25762,0,0,25762,55.01
30,individualCount,17697,0,0,17697,37.79
23,locality,15251,38,0,15289,32.65
35,collectionCode,13469,0,0,13469,28.76


,grupimi,grupi,n,zero_count,elevation_present,zero_pct_group,zero_pct_present,datasetTitle
95,datasetKey,bc3b0bf0-e245-11dd-ab80-b8a03c50a862,50,10,50,20.00,20.00,World flea collection of Slovenian Museum of Natural History (excluding Slovenia)
110,datasetKey,d92e92e0-8d20-4756-a60b-3746f408c14c,8107,3624,8107,44.70,44.70,Arthropod observations derived from HabiProt Alciphron database in Serbia
125,basisOfRecord,HUMAN_OBSERVATION,42479,3624,8696,8.53,41.67,
131,basisOfRecord,PRESERVED_SPECIMEN,1748,10,1013,0.57,0.99,
132,district,Ferizaj / Uroševac,2593,491,913,18.94,53.78,
133,district,Gjakovë / Đakovica,2697,473,1043,17.54,45.35,
134,district,Gjilan / Gnjilane,4156,9,2285,0.22,0.39,
135,district,Mitrovicë / Kosovska Mitrovica,3196,590,1819,18.46,32.44,
136,district,Pejë / Peć,3187,474,727,14.87,65.20,
137,district,Prishtinë / Priština,25734,678,1151,2.63,58.91,


# Vendimi për elevation = 0

Ka 3,634 zero (7.76% të rreshtave; 37.01% të 9,820 lartësive të plotësuara).
Burimi kryesor është Arthropod observations derived from HabiProt Alciphron database in Serbia (`d92e92e0-8d20-4756-a60b-3746f408c14c`), me 3,624 zero.
Zerot shfaqen në 7 rajone dhe 32 komuna; nuk përfaqësojnë një vend të vetëm të ulët.
Tabelat `elevation_zero_by_group.csv` japin numrin dhe emëruesin sipas burimit, basisOfRecord, rajonit, komunës dhe locality; `elevation_zero_records.csv` ruan koordinatat dhe ID-të për auditim.

**Decision + Rationale:** 0 m nuk trajtohet si lartësi reale e sipërfaqes në Kosovë. ASK raporton minimum 270 m në luginën e Drinit të Bardhë ([Some Facts on the Environment 2021–2022, f. 10](https://askapi.rks-gov.net/Custom/d9b7ae43-c68e-49a4-b713-5ec33c3a9c25.pdf#page=10)). Përqendrimi te një burim sugjeron placeholder ose problem eksportimi, por pa metadata të botuesit nuk mund të provohet shkaku për çdo rresht.

Në analizën e mungesave zero përfshihet në maskën efektive; në pastrimin e ardhshëm rekomandohet NaN dhe flamur `elevation_was_zero`, duke ruajtur origjinalin. Nuk zëvendësohet me 270 m ose me mesatare. Mungesa efektive bëhet 40,645 (86.79%).

Kufizime: nuk është bërë kontroll GIS/DEM për çdo pikë; lartësitë e tjera të dyshimta (përfshirë maksimumin 3050 m) kërkojnë kontroll të veçantë. Referenca kombëtare nuk është prag automatik për të fshirë të gjitha vlerat nën 270 m. Këtu krijohen vetëm raporte, jo dataset i pastruar.


### Dokumentimi i hapit: Mungesat e fshehura

1. **Input:** `dataset/kosovo_overall_biodiversity.csv`; inventari NaN dhe raportet e cituara në këtë seksion.
2. **Çka u bë:** U audituan tokenët literalë dhe shpërndarja e elevation = 0.
3. **Rreshta para / pas:** 46,831 / 46,831; **kolona para / pas:** 42 / 42. Maskat diagnostikuese janë të ndara nga dataset-i.
4. **Pse u bë:** Për të mbështetur trajtimin e mungesave me evidencë dhe kuptim semantik.
5. **Output:** `reports/hidden_missing_values.csv`, `effective_missing_report.csv`, `elevation_zero_by_group.csv`, `elevation_zero_records.csv`, `elevation_zero_decision.md`.
6. **Commit:** `Add hidden missingness analysis and column strategies` (mesazh i propozuar).
7. **Decision + Rationale:** Zero në elevation përfshihet si mungesë efektive, e arsyetuar në raport; tokenët numërohen pa dyfishuar NaN. Nuk bëhen zëvendësime në df_raw.

## Mekanizmat e mungesës

Përdoren `datasetKey` si burim, `basisOfRecord`, `class` dhe `district` si rajon. `coordinateUncertainty` i detyrës i referohet `coordinateUncertaintyInMeters`. Çdo përqindje ka emërues numrin e rreshtave të grupit, jo të të gjithë dataset-it. Krahasohen NaN fillestar dhe mungesat efektive, përfshirë elevation = 0. Grupet me çelës të zbrazët ruhen si `[mungon]`.

In [7]:
section("Mekanizmat e mungesës")
targets = ["elevation", "individualCount", "coordinateUncertaintyInMeters", "eventDate", "species"]
groupings = ["datasetKey", "basisOfRecord", "class", "district"]
group_tables = []
for grouping in groupings:
    labels = group_labels(grouping)
    for column in targets:
        frame = pd.DataFrame({"grupi": labels, "raw": df_raw[column].isna(), "effective": effective_missing[column]})
        table = frame.groupby("grupi", dropna=False).agg(
            n=("raw", "size"), nan_fillestar=("raw", "sum"), mungesa_efektive=("effective", "sum")
        ).reset_index()
        table.insert(0, "kolona", column)
        table.insert(0, "grupimi", grouping)
        table["pct_nan"] = (table["nan_fillestar"] / table["n"] * 100).round(2)
        table["pct_efektive"] = (table["mungesa_efektive"] / table["n"] * 100).round(2)
        table["grup_i_vogel"] = table["n"] < 30
        table["datasetTitle"] = table["grupi"].map(source_titles) if grouping == "datasetKey" else ""
        group_tables.append(table)
missing_by_group = pd.concat(group_tables, ignore_index=True)
missing_by_group.to_csv(REPORTS_DIR / "missingness_by_group.csv", index=False)
display(missing_by_group[missing_by_group["grupimi"].eq("basisOfRecord")])

interpretations = []
for column in targets:
    eligible = missing_by_group[(missing_by_group["kolona"] == column) & (missing_by_group["n"] >= 30)]
    descriptions = []
    for grouping in groupings:
        subset = eligible[eligible["grupimi"] == grouping]
        if len(subset) >= 2:
            low = subset.loc[subset["pct_efektive"].idxmin()]
            high = subset.loc[subset["pct_efektive"].idxmax()]
            descriptions.append(f"{grouping}: {low['pct_efektive']:.2f}% ({low['grupi']}, n={low['n']}) deri {high['pct_efektive']:.2f}% ({high['grupi']}, n={high['n']})")
    interpretations.append(f"- **{column}**: " + "; ".join(descriptions) + ".")
mechanisms_report = """# Mekanizmat e mungesës

Përqindjet bazohen në maskat efektive të seksionit paraprak. CSV-ja ruan edhe numërimet dhe përqindjet origjinale NaN. Grupimet janë margjinale, jo kombinime të katër faktorëve. Të gjitha grupet ruhen në CSV; më poshtë krahasohen vetëm grupet me n >= 30 për të mos mbështetur interpretimin te grupe tepër të vogla. Ky kufi është vetëm për përmbledhjen, jo prag imputimi apo test statistikor.

""" + "\n\n".join(interpretations) + """

**Decision + Rationale:** Përqindjet ndryshojnë sipas karakteristikave të vëzhguara. The missingness pattern appears consistent with MAR conditional on observed source and record characteristics, but MAR is not established by these descriptive comparisons. Prandaj MCAR nuk merret si supozim pune dhe nuk përdoret imputim i përgjithshëm me mesatare ose modë.

Kufizime: tabelat nuk provojnë MAR dhe nuk përjashtojnë MNAR; vlerat e pavëzhguara nuk mund të testohen nga ky CSV. Burimi, klasa, rajoni dhe basisOfRecord janë të ndërvarura; nuk interpretohen si shkaqe të pavarura. Vëzhgimet nuk janë mostër e rastësishme dhe mund të kenë varësi hapësinore, kohore ose dublikime. Për species mungesa mund të lidhet me rangun taksonomik; për individualCount me mënyrën e raportimit; për elevation me konventat e burimit. Këto janë shpjegime të mundshme, jo mekanizma të provuar. Datat e pjesshme të plotësuara nuk janë NaN: precision duhet ruajtur veçmas në pastrim. Nevojiten metadata të burimeve dhe analizë ndjeshmërie para imputimit modelor.
"""
(REPORTS_DIR / "missingness_mechanisms.md").write_text(mechanisms_report, encoding="utf-8")
display(Markdown(mechanisms_report))


Mekanizmat e mungesës


,grupimi,kolona,grupi,n,nan_fillestar,mungesa_efektive,pct_nan,pct_efektive,grup_i_vogel,datasetTitle
625,basisOfRecord,elevation,HUMAN_OBSERVATION,42479,33783,37407,79.53,88.06,False,
626,basisOfRecord,elevation,MACHINE_OBSERVATION,376,376,376,100.00,100.00,False,
627,basisOfRecord,elevation,MATERIAL_CITATION,317,292,292,92.11,92.11,False,
628,basisOfRecord,elevation,MATERIAL_SAMPLE,641,605,605,94.38,94.38,False,
629,basisOfRecord,elevation,OBSERVATION,1,0,0,0.00,0.00,True,
630,basisOfRecord,elevation,OCCURRENCE,1269,1220,1220,96.14,96.14,False,
631,basisOfRecord,elevation,PRESERVED_SPECIMEN,1748,735,745,42.05,42.62,False,
632,basisOfRecord,individualCount,HUMAN_OBSERVATION,42479,13828,13828,32.55,32.55,False,
633,basisOfRecord,individualCount,MACHINE_OBSERVATION,376,363,363,96.54,96.54,False,
634,basisOfRecord,individualCount,MATERIAL_CITATION,317,100,100,31.55,31.55,False,


# Mekanizmat e mungesës

Përqindjet bazohen në maskat efektive të seksionit paraprak. CSV-ja ruan edhe numërimet dhe përqindjet origjinale NaN. Grupimet janë margjinale, jo kombinime të katër faktorëve. Të gjitha grupet ruhen në CSV; më poshtë krahasohen vetëm grupet me n >= 30 për të mos mbështetur interpretimin te grupe tepër të vogla. Ky kufi është vetëm për përmbledhjen, jo prag imputimi apo test statistikor.

- **elevation**: datasetKey: 0.00% (03269e13-84ae-430f-990e-f11069413e36, n=52) deri 100.00% (07044577-bd82-4089-9f3a-f4a9d2170b2e, n=386); basisOfRecord: 42.62% (PRESERVED_SPECIMEN, n=1748) deri 100.00% (MACHINE_OBSERVATION, n=376); class: 3.76% (Lecanoromycetes, n=612) deri 100.00% (Amphibia, n=109); district: 45.24% (Gjilan / Gnjilane, n=4156) deri 98.16% (Prishtinë / Priština, n=25734).

- **individualCount**: datasetKey: 0.00% (03269e13-84ae-430f-990e-f11069413e36, n=52) deri 100.00% (040c5662-da76-4782-a48e-cdea1892d14c, n=341); basisOfRecord: 31.55% (MATERIAL_CITATION, n=317) deri 96.54% (MACHINE_OBSERVATION, n=376); class: 10.63% (Aves, n=28785) deri 100.00% (Bivalvia, n=40); district: 15.00% (Prishtinë / Priština, n=25734) deri 71.87% (Prizren, n=5268).

- **coordinateUncertaintyInMeters**: datasetKey: 0.00% (6ac3f774-d9fb-4796-b3e9-92bf6c81c084, n=721) deri 100.00% (03269e13-84ae-430f-990e-f11069413e36, n=52); basisOfRecord: 66.57% (HUMAN_OBSERVATION, n=42479) deri 97.79% (MATERIAL_CITATION, n=317); class: 11.36% (Mammalia, n=273) deri 100.00% (Clitellata, n=43); district: 30.38% (Mitrovicë / Kosovska Mitrovica, n=3196) deri 87.13% (Prishtinë / Priština, n=25734).

- **eventDate**: datasetKey: 0.00% (03269e13-84ae-430f-990e-f11069413e36, n=52) deri 100.00% (69bbc9fb-a432-4945-bb4f-654bff3bc2de, n=46); basisOfRecord: 0.00% (MACHINE_OBSERVATION, n=376) deri 52.01% (OCCURRENCE, n=1269); class: 0.00% (Agaricomycetes, n=187) deri 95.00% (Bivalvia, n=40); district: 0.85% (Prishtinë / Priština, n=25734) deri 16.49% (Mitrovicë / Kosovska Mitrovica, n=3196).

- **species**: datasetKey: 0.00% (03269e13-84ae-430f-990e-f11069413e36, n=52) deri 94.51% (bd14614c-4d72-4763-8710-1cea514311cc, n=182); basisOfRecord: 0.00% (MACHINE_OBSERVATION, n=376) deri 55.23% (MATERIAL_SAMPLE, n=641); class: 0.00% (Amphibia, n=109) deri 62.50% (Bivalvia, n=40); district: 1.44% (Gjilan / Gnjilane, n=4156) deri 7.66% (Pejë / Peć, n=3187).

**Decision + Rationale:** Përqindjet ndryshojnë sipas karakteristikave të vëzhguara. The missingness pattern appears consistent with MAR conditional on observed source and record characteristics, but MAR is not established by these descriptive comparisons. Prandaj MCAR nuk merret si supozim pune dhe nuk përdoret imputim i përgjithshëm me mesatare ose modë.

Kufizime: tabelat nuk provojnë MAR dhe nuk përjashtojnë MNAR; vlerat e pavëzhguara nuk mund të testohen nga ky CSV. Burimi, klasa, rajoni dhe basisOfRecord janë të ndërvarura; nuk interpretohen si shkaqe të pavarura. Vëzhgimet nuk janë mostër e rastësishme dhe mund të kenë varësi hapësinore, kohore ose dublikime. Për species mungesa mund të lidhet me rangun taksonomik; për individualCount me mënyrën e raportimit; për elevation me konventat e burimit. Këto janë shpjegime të mundshme, jo mekanizma të provuar. Datat e pjesshme të plotësuara nuk janë NaN: precision duhet ruajtur veçmas në pastrim. Nevojiten metadata të burimeve dhe analizë ndjeshmërie para imputimit modelor.


### Dokumentimi i hapit: Mekanizmat e mungesës

1. **Input:** `dataset/kosovo_overall_biodiversity.csv`; inventari NaN dhe raportet e cituara në këtë seksion.
2. **Çka u bë:** U llogaritën numërimet, emëruesit dhe përqindjet për pesë kolona sipas katër grupimeve.
3. **Rreshta para / pas:** 46,831 / 46,831; **kolona para / pas:** 42 / 42. Maskat diagnostikuese janë të ndara nga dataset-i.
4. **Pse u bë:** Për të mbështetur trajtimin e mungesave me evidencë dhe kuptim semantik.
5. **Output:** `reports/missingness_by_group.csv`, `reports/missingness_mechanisms.md`.
6. **Commit:** `Add hidden missingness analysis and column strategies` (mesazh i propozuar).
7. **Decision + Rationale:** MAR është vetëm supozim i mundshëm i kushtëzuar; krahasimet përshkruese nuk e provojnë dhe nuk përjashtojnë MNAR.

## Strategjia për vlerat e zbrazëta

Rëndësia analitike merret nga `reports/column_importance.md`; vendimi për redundancën nga `reports/redundancy.md`. Strategjia është plan për fazën e pastrimit: `delete` nënkupton heqje vetëm nga pamja analitike e përcaktuar, jo nga origjinali. `impute` më poshtë nënkupton rikuperim determinist të verifikuar, jo krijim vlerash me mesatare/modë. Nuk përdoret prag i përgjithshëm i mungesës.

In [8]:
section("Strategjia për vlerat e zbrazëta")
import re

importance_text = (REPORTS_DIR / "column_importance.md").read_text(encoding="utf-8")
importance = dict(re.findall(r"^\| `([^`]+)` \| [^|]+ \| ([^|]+) \|$", importance_text, re.MULTILINE))
assert set(df_raw.columns) <= importance.keys(), "Mungon vlerësimi i rëndësisë për disa kolona."
strategies = {}

def strategy(columns, recovery, decision, reason):
    for column in columns.split():
        strategies[column] = (recovery, decision, reason)

strategy("gbifID datasetKey", "Vetëm nga burimi origjinal; jo gjenerim ID-sh.", "keep", "Identifikues për gjurmueshmëri dhe grupim; ruhet edhe kur rëndësia direkte analitike është e ulët.")
strategy("scientificName taxonRank", "Validim me taksonominë e burimit.", "keep + flag", "Ruhen emri dhe rangu real; mungesa nuk zëvendësohet me taksonin më të shpeshtë.")
strategy("species speciesKey", "Vetëm lidhje e verifikuar me speciesKey/taksonominë.", "keep + flag", "Mos shpik species nga gjinia. Përjashto vetëm nga analiza në nivel species, jo nga dataset-i i përgjithshëm.")
strategy("kingdom phylum class order family genus taxonKey", "Hierarki taksonomike e verifikuar nga çelësi GBIF.", "keep + flag", "Ruaj nivelin e identifikimit; rikuperim vetëm kur lidhja është e qartë dhe dokumentohet.")
strategy("vernacularName", "Fjalor taksonomik me gjuhë të përcaktuar.", "keep", "Emri i zakonshëm është opsional; mungesa nuk pengon identifikimin shkencor.")
strategy("iucnRedListCategory", "Vetëm vlerësim IUCN i verifikuar për taksonin dhe versionin.", "keep + flag", "NE dhe DD janë kategori reale, jo NaN dhe jo LC; mos imputo status rreziku.")
strategy("iucnRedListStatus", "Rindërtohet nga harta e kodeve të IUCN.", "delete (pamja analitike)", "Redundant sipas reports/redundancy.md; jo vendim i bazuar në përqindjen e mungesës.")
strategy("eventDate", "Komponentë year/month/day vetëm nëse janë të vlefshëm e konsistentë.", "keep + impute + flag", "Ruaj intervalet dhe precision; mos krijo muaj/ditë nga data vetëm me vit. Rikuperimi i ardhshëm shënohet.")
strategy("year month day", "Nga eventDate vetëm kur komponenti është shprehimisht i pranishëm dhe i vlefshëm.", "keep + impute + flag", "Mos përdor 1 janar si datë reale; rreshtat pa komponentin e nevojshëm përjashtohen vetëm nga analiza kohore përkatëse.")
strategy("decimalLatitude decimalLongitude", "Vetëm gjeoreferencim i dokumentuar.", "keep + flag", "Pa mesatare koordinatash; ruaj pikën dhe verifiko vlefshmërinë/hartën veçmas.")
strategy("coordinateUncertaintyInMeters", "Metadata e gjeoreferencimit nga burimi.", "keep + flag", "Mungesa nuk është pasiguri zero. Analizat hapësinore duhet të raportojnë filtrat e cilësisë.")
strategy("elevation", "DEM i verifikuar nga koordinatat, me rezolucion dhe proveniencë; ende jo zbatuar.", "keep + flag", "Zero trajtohet si NaN në kopjen e ardhshme me elevation_was_zero; pa medianë globale. Përdorim vetëm pas kontrollit të plausibilitetit.")
strategy("locality", "Burimi origjinal; komuna nuk zëvendëson lokalitetin.", "keep + flag", "unknown dhe tokenët bosh normalizohen në kopjen e pastruar; mos shpik lokalitet specifik.")
strategy("municipality district", "Spatial join me kufij administrativë të verifikuar nëse nevojitet.", "keep", "Kolona të plota për analizë rajonale; harmonizimi i emrave është hap i veçantë.")
strategy("protectedArea protectedAreaDesignation", "Spatial join me kufijtë dhe versionin e zonave të mbrojtura.", "keep + flag", "Raportet ekzistuese sugjerojnë jashtë zonës, por mungesa vetëm nuk provon Not protected. Mbaj unknown derisa mbulimi hapësinor të verifikohet.")
strategy("strictlyProtected", "Metadata/hartë e mbrojtjes.", "keep", "False është vlerë e vlefshme; mungesa eventuale nuk zëvendësohet automatikisht me False.")
strategy("basisOfRecord", "Metadata e burimit.", "keep", "Përcakton kuptimin e regjistrimit dhe dallimet e mungesës; nuk imputohet me modë.")
strategy("individualCount", "Vetëm numër i raportuar nga burimi.", "keep + flag", "Mungesa nuk është 0 ose 1. Mos ngatërro numrin e regjistrimeve me numrin e individëve; pa imputim global.")
strategy("establishmentMeans", "Informacion ekologjik i verifikuar.", "delete (pamja kryesore)", "Pothuajse bosh dhe jashtë pyetjeve kryesore sipas rëndësisë; ruaj origjinalin për analizë të specializuar.")
strategy("recordedBy identifiedBy", "Metadata e regjistrimit.", "keep + flag", "unknown normalizohet; mos imputo identitetin me emrin më të shpeshtë.")
strategy("institutionCode collectionCode catalogNumber", "Metadata e institucionit/koleksionit për regjistrimin.", "keep", "Fusha provenience opsionale; datasetKey nuk përcakton gjithmonë një institucion, prandaj jo mbushje e verbër.")
strategy("license", "Licenca e regjistrimit/burimit.", "keep", "Duhet për përdorim dhe atribuim pavarësisht rëndësisë së ulët analitike.")
strategy("issue", "Flag-et e eksportit GBIF.", "keep + flag", "Bosh mund të nënkuptojë pa flag të raportuar, jo cilësi të garantuar; mos imputo problem.")
strategy("iasUnionConcern", "Lista dhe metadata për versionin përkatës.", "keep", "False është vlerë e vlefshme; nuk tregon domosdoshmërisht plotësinë e njohurive ekologjike.")
strategy("iasAppliesFrom", "Data e listimit vetëm kur zbatohet iasUnionConcern.", "keep + flag", "Mungesa mund të jetë strukturore për False. Ndaj not applicable nga unknown kur True; mos hiq vetëm për shkak të 99.95% mungesës.")
assert set(strategies) == set(df_raw.columns)

def markdown_table(frame):
    def escape(value):
        return str(value).replace("|", "\\|").replace("\n", " ")
    lines = ["| " + " | ".join(map(escape, frame.columns)) + " |", "| " + " | ".join(["---"] * len(frame.columns)) + " |"]
    lines.extend("| " + " | ".join(map(escape, row)) + " |" for row in frame.itertuples(index=False, name=None))
    return "\n".join(lines)

strategy_rows = []
for column in df_raw:
    recovery, decision, reason = strategies[column]
    strategy_rows.append({"Column": column, "Missing %": f"{effective_missing[column].mean() * 100:.2f}",
                          "Rëndësia": importance[column].strip(), "Rikuperimi": recovery,
                          "Vendimi": decision, "Arsyetimi": reason})
strategy_table = pd.DataFrame(strategy_rows)
ias_check = pd.crosstab(literal["iasUnionConcern"], effective_missing["iasAppliesFrom"], dropna=False)
strategy_report = """# Strategjia për vlerat e zbrazëta

Input: dataset-i origjinal, `effective_missing_report.csv`, `missingness_by_group.csv`, `missingness_mechanisms.md`, `column_importance.md` dhe `redundancy.md`.

**Missing %** është përqindja efektive: bashkimi i NaN, tokenëve literalë të mungesës dhe elevation = 0. Inventari fillestar mbetet në `processed/missing_report.csv`; numërimet e krahasueshme janë në `reports/effective_missing_report.csv`. NE/DD janë kategori IUCN; mungesat potencialisht strukturore ende numërohen dhe interpretohen veçmas.

**Decision + Rationale:** Nuk ka prag fiks. Vendimet mbështeten në rëndësi, kuptim, mundësi rikuperimi dhe modelin e mungesës. Delete vlen vetëm për pamjen e specifikuar; keep ruan vlerën dhe NaN; flag shton në fazën e zbatimit një tregues mungese; impute lejohet vetëm për rikuperim determinist të verifikuar. Asnjë vendim në tabelë nuk është zbatuar mbi dataset-in. Nuk ka fshirje globale rreshtash për species/year. Përqindjet e larta nuk justifikojnë vetë heqjen.

""" + markdown_table(strategy_table) + """

## Kontrolli i mungesës strukturore

iasAppliesFrom sipas iasUnionConcern (False/True në kolonat më poshtë tregojnë nëse data mungon):

""" + markdown_table(ias_check.reset_index()) + """

Për protectedArea raporti i mëparshëm e interpreton boshllëkun si jashtë zonave të mbrojtura. Këtu kjo mbetet hipotezë deri në verifikimin e mbulimit dhe versionit të spatial join; nuk caktohet automatikisht Not protected.

## Dokumentimi i detyrës

- **Input:** dataset/kosovo_overall_biodiversity.csv dhe raportet e mësipërme.
- **Çka u bë:** U vendos strategji për të 42 kolonat me rëndësi, rikuperim dhe arsyetim.
- **Rreshta para / pas:** 46,831 / 46,831; **kolona para / pas:** 42 / 42.
- **Pse u bë:** Për të shmangur humbje informacioni dhe imputime pa mbështetje semantike.
- **Output:** reports/missing_strategy.md, 42 vendime.
- **Commit:** Add hidden missingness analysis and column strategies (mesazh i propozuar).

Këto janë hapa diagnostikimi dhe planifikimi; cleaning_log.csv ruhet për transformimet reale. Zbatimi i vendimeve duhet të regjistrojë ndryshimet me log_step().
"""
(REPORTS_DIR / "missing_strategy.md").write_text(strategy_report, encoding="utf-8")
display(Markdown(strategy_report))


Strategjia për vlerat e zbrazëta


# Strategjia për vlerat e zbrazëta

Input: dataset-i origjinal, `effective_missing_report.csv`, `missingness_by_group.csv`, `missingness_mechanisms.md`, `column_importance.md` dhe `redundancy.md`.

**Missing %** është përqindja efektive: bashkimi i NaN, tokenëve literalë të mungesës dhe elevation = 0. Inventari fillestar mbetet në `processed/missing_report.csv`; numërimet e krahasueshme janë në `reports/effective_missing_report.csv`. NE/DD janë kategori IUCN; mungesat potencialisht strukturore ende numërohen dhe interpretohen veçmas.

**Decision + Rationale:** Nuk ka prag fiks. Vendimet mbështeten në rëndësi, kuptim, mundësi rikuperimi dhe modelin e mungesës. Delete vlen vetëm për pamjen e specifikuar; keep ruan vlerën dhe NaN; flag shton në fazën e zbatimit një tregues mungese; impute lejohet vetëm për rikuperim determinist të verifikuar. Asnjë vendim në tabelë nuk është zbatuar mbi dataset-in. Nuk ka fshirje globale rreshtash për species/year. Përqindjet e larta nuk justifikojnë vetë heqjen.

| Column | Missing % | Rëndësia | Rikuperimi | Vendimi | Arsyetimi |
| --- | --- | --- | --- | --- | --- |
| gbifID | 0.00 | e ulët | Vetëm nga burimi origjinal; jo gjenerim ID-sh. | keep | Identifikues për gjurmueshmëri dhe grupim; ruhet edhe kur rëndësia direkte analitike është e ulët. |
| scientificName | 0.08 | e lartë | Validim me taksonominë e burimit. | keep + flag | Ruhen emri dhe rangu real; mungesa nuk zëvendësohet me taksonin më të shpeshtë. |
| species | 2.59 | e lartë | Vetëm lidhje e verifikuar me speciesKey/taksonominë. | keep + flag | Mos shpik species nga gjinia. Përjashto vetëm nga analiza në nivel species, jo nga dataset-i i përgjithshëm. |
| vernacularName | 8.27 | e ulët | Fjalor taksonomik me gjuhë të përcaktuar. | keep | Emri i zakonshëm është opsional; mungesa nuk pengon identifikimin shkencor. |
| kingdom | 0.08 | e lartë | Hierarki taksonomike e verifikuar nga çelësi GBIF. | keep + flag | Ruaj nivelin e identifikimit; rikuperim vetëm kur lidhja është e qartë dhe dokumentohet. |
| phylum | 0.15 | e lartë | Hierarki taksonomike e verifikuar nga çelësi GBIF. | keep + flag | Ruaj nivelin e identifikimit; rikuperim vetëm kur lidhja është e qartë dhe dokumentohet. |
| class | 0.83 | e lartë | Hierarki taksonomike e verifikuar nga çelësi GBIF. | keep + flag | Ruaj nivelin e identifikimit; rikuperim vetëm kur lidhja është e qartë dhe dokumentohet. |
| order | 0.71 | e lartë | Hierarki taksonomike e verifikuar nga çelësi GBIF. | keep + flag | Ruaj nivelin e identifikimit; rikuperim vetëm kur lidhja është e qartë dhe dokumentohet. |
| family | 0.34 | e lartë | Hierarki taksonomike e verifikuar nga çelësi GBIF. | keep + flag | Ruaj nivelin e identifikimit; rikuperim vetëm kur lidhja është e qartë dhe dokumentohet. |
| genus | 0.42 | e lartë | Hierarki taksonomike e verifikuar nga çelësi GBIF. | keep + flag | Ruaj nivelin e identifikimit; rikuperim vetëm kur lidhja është e qartë dhe dokumentohet. |
| taxonRank | 0.08 | e mesme | Validim me taksonominë e burimit. | keep + flag | Ruhen emri dhe rangu real; mungesa nuk zëvendësohet me taksonin më të shpeshtë. |
| taxonKey | 0.08 | e mesme | Hierarki taksonomike e verifikuar nga çelësi GBIF. | keep + flag | Ruaj nivelin e identifikimit; rikuperim vetëm kur lidhja është e qartë dhe dokumentohet. |
| speciesKey | 2.59 | e mesme | Vetëm lidhje e verifikuar me speciesKey/taksonominë. | keep + flag | Mos shpik species nga gjinia. Përjashto vetëm nga analiza në nivel species, jo nga dataset-i i përgjithshëm. |
| iucnRedListCategory | 2.59 | e lartë | Vetëm vlerësim IUCN i verifikuar për taksonin dhe versionin. | keep + flag | NE dhe DD janë kategori reale, jo NaN dhe jo LC; mos imputo status rreziku. |
| iucnRedListStatus | 2.59 | e ulët | Rindërtohet nga harta e kodeve të IUCN. | delete (pamja analitike) | Redundant sipas reports/redundancy.md; jo vendim i bazuar në përqindjen e mungesës. |
| eventDate | 4.68 | e lartë | Komponentë year/month/day vetëm nëse janë të vlefshëm e konsistentë. | keep + impute + flag | Ruaj intervalet dhe precision; mos krijo muaj/ditë nga data vetëm me vit. Rikuperimi i ardhshëm shënohet. |
| year | 4.84 | e lartë | Nga eventDate vetëm kur komponenti është shprehimisht i pranishëm dhe i vlefshëm. | keep + impute + flag | Mos përdor 1 janar si datë reale; rreshtat pa komponentin e nevojshëm përjashtohen vetëm nga analiza kohore përkatëse. |
| month | 6.12 | e lartë | Nga eventDate vetëm kur komponenti është shprehimisht i pranishëm dhe i vlefshëm. | keep + impute + flag | Mos përdor 1 janar si datë reale; rreshtat pa komponentin e nevojshëm përjashtohen vetëm nga analiza kohore përkatëse. |
| day | 6.87 | e mesme | Nga eventDate vetëm kur komponenti është shprehimisht i pranishëm dhe i vlefshëm. | keep + impute + flag | Mos përdor 1 janar si datë reale; rreshtat pa komponentin e nevojshëm përjashtohen vetëm nga analiza kohore përkatëse. |
| decimalLatitude | 0.00 | e mesme | Vetëm gjeoreferencim i dokumentuar. | keep + flag | Pa mesatare koordinatash; ruaj pikën dhe verifiko vlefshmërinë/hartën veçmas. |
| decimalLongitude | 0.00 | e mesme | Vetëm gjeoreferencim i dokumentuar. | keep + flag | Pa mesatare koordinatash; ruaj pikën dhe verifiko vlefshmërinë/hartën veçmas. |
| coordinateUncertaintyInMeters | 68.34 | e lartë | Metadata e gjeoreferencimit nga burimi. | keep + flag | Mungesa nuk është pasiguri zero. Analizat hapësinore duhet të raportojnë filtrat e cilësisë. |
| elevation | 86.79 | e ulët | DEM i verifikuar nga koordinatat, me rezolucion dhe proveniencë; ende jo zbatuar. | keep + flag | Zero trajtohet si NaN në kopjen e ardhshme me elevation_was_zero; pa medianë globale. Përdorim vetëm pas kontrollit të plausibilitetit. |
| locality | 32.65 | e mesme | Burimi origjinal; komuna nuk zëvendëson lokalitetin. | keep + flag | unknown dhe tokenët bosh normalizohen në kopjen e pastruar; mos shpik lokalitet specifik. |
| municipality | 0.00 | e lartë | Spatial join me kufij administrativë të verifikuar nëse nevojitet. | keep | Kolona të plota për analizë rajonale; harmonizimi i emrave është hap i veçantë. |
| district | 0.00 | e mesme | Spatial join me kufij administrativë të verifikuar nëse nevojitet. | keep | Kolona të plota për analizë rajonale; harmonizimi i emrave është hap i veçantë. |
| protectedArea | 55.01 | e lartë | Spatial join me kufijtë dhe versionin e zonave të mbrojtura. | keep + flag | Raportet ekzistuese sugjerojnë jashtë zonës, por mungesa vetëm nuk provon Not protected. Mbaj unknown derisa mbulimi hapësinor të verifikohet. |
| protectedAreaDesignation | 55.01 | e lartë | Spatial join me kufijtë dhe versionin e zonave të mbrojtura. | keep + flag | Raportet ekzistuese sugjerojnë jashtë zonës, por mungesa vetëm nuk provon Not protected. Mbaj unknown derisa mbulimi hapësinor të verifikohet. |
| strictlyProtected | 0.00 | e mesme | Metadata/hartë e mbrojtjes. | keep | False është vlerë e vlefshme; mungesa eventuale nuk zëvendësohet automatikisht me False. |
| basisOfRecord | 0.00 | e lartë | Metadata e burimit. | keep | Përcakton kuptimin e regjistrimit dhe dallimet e mungesës; nuk imputohet me modë. |
| individualCount | 37.79 | e ulët | Vetëm numër i raportuar nga burimi. | keep + flag | Mungesa nuk është 0 ose 1. Mos ngatërro numrin e regjistrimeve me numrin e individëve; pa imputim global. |
| establishmentMeans | 99.49 | e ulët | Informacion ekologjik i verifikuar. | delete (pamja kryesore) | Pothuajse bosh dhe jashtë pyetjeve kryesore sipas rëndësisë; ruaj origjinalin për analizë të specializuar. |
| recordedBy | 8.23 | e ulët | Metadata e regjistrimit. | keep + flag | unknown normalizohet; mos imputo identitetin me emrin më të shpeshtë. |
| identifiedBy | 83.49 | e ulët | Metadata e regjistrimit. | keep + flag | unknown normalizohet; mos imputo identitetin me emrin më të shpeshtë. |
| institutionCode | 7.92 | e mesme | Metadata e institucionit/koleksionit për regjistrimin. | keep | Fusha provenience opsionale; datasetKey nuk përcakton gjithmonë një institucion, prandaj jo mbushje e verbër. |
| collectionCode | 28.76 | e ulët | Metadata e institucionit/koleksionit për regjistrimin. | keep | Fusha provenience opsionale; datasetKey nuk përcakton gjithmonë një institucion, prandaj jo mbushje e verbër. |
| catalogNumber | 26.82 | e ulët | Metadata e institucionit/koleksionit për regjistrimin. | keep | Fusha provenience opsionale; datasetKey nuk përcakton gjithmonë një institucion, prandaj jo mbushje e verbër. |
| datasetKey | 0.00 | e mesme | Vetëm nga burimi origjinal; jo gjenerim ID-sh. | keep | Identifikues për gjurmueshmëri dhe grupim; ruhet edhe kur rëndësia direkte analitike është e ulët. |
| license | 0.00 | e ulët | Licenca e regjistrimit/burimit. | keep | Duhet për përdorim dhe atribuim pavarësisht rëndësisë së ulët analitike. |
| issue | 3.04 | e lartë | Flag-et e eksportit GBIF. | keep + flag | Bosh mund të nënkuptojë pa flag të raportuar, jo cilësi të garantuar; mos imputo problem. |
| iasUnionConcern | 0.00 | e ulët | Lista dhe metadata për versionin përkatës. | keep | False është vlerë e vlefshme; nuk tregon domosdoshmërisht plotësinë e njohurive ekologjike. |
| iasAppliesFrom | 99.95 | e ulët | Data e listimit vetëm kur zbatohet iasUnionConcern. | keep + flag | Mungesa mund të jetë strukturore për False. Ndaj not applicable nga unknown kur True; mos hiq vetëm për shkak të 99.95% mungesës. |

## Kontrolli i mungesës strukturore

iasAppliesFrom sipas iasUnionConcern (False/True në kolonat më poshtë tregojnë nëse data mungon):

| iasUnionConcern | False | True |
| --- | --- | --- |
| FALSE | 0 | 46807 |
| TRUE | 24 | 0 |

Për protectedArea raporti i mëparshëm e interpreton boshllëkun si jashtë zonave të mbrojtura. Këtu kjo mbetet hipotezë deri në verifikimin e mbulimit dhe versionit të spatial join; nuk caktohet automatikisht Not protected.

## Dokumentimi i detyrës

- **Input:** dataset/kosovo_overall_biodiversity.csv dhe raportet e mësipërme.
- **Çka u bë:** U vendos strategji për të 42 kolonat me rëndësi, rikuperim dhe arsyetim.
- **Rreshta para / pas:** 46,831 / 46,831; **kolona para / pas:** 42 / 42.
- **Pse u bë:** Për të shmangur humbje informacioni dhe imputime pa mbështetje semantike.
- **Output:** reports/missing_strategy.md, 42 vendime.
- **Commit:** Add hidden missingness analysis and column strategies (mesazh i propozuar).

Këto janë hapa diagnostikimi dhe planifikimi; cleaning_log.csv ruhet për transformimet reale. Zbatimi i vendimeve duhet të regjistrojë ndryshimet me log_step().


### Dokumentimi i hapit: Strategjia e mungesave

1. **Input:** `dataset/kosovo_overall_biodiversity.csv`; inventari NaN dhe raportet e cituara në këtë seksion.
2. **Çka u bë:** U përgatit vendimi për secilën nga 42 kolonat, duke përdorur rëndësinë analitike të Jetës.
3. **Rreshta para / pas:** 46,831 / 46,831; **kolona para / pas:** 42 / 42. Maskat diagnostikuese janë të ndara nga dataset-i.
4. **Pse u bë:** Për të mbështetur trajtimin e mungesave me evidencë dhe kuptim semantik.
5. **Output:** `reports/missing_strategy.md` (42 rreshta strategjie).
6. **Commit:** `Add hidden missingness analysis and column strategies` (mesazh i propozuar).
7. **Decision + Rationale:** Pa prag të përgjithshëm; rikuperimi determinist është i kushtëzuar nga verifikimi. Vendimet nuk janë zbatuar ende si transformime.